# graphforest — IEEE-CIS benchmark
Real fraud detection on the IEEE-CIS dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.model_selection import train_test_split
from graphforest import DirectedGraphForestClassifier, GraphForestExplainer
print('imports OK')

In [ ]:
# Load and merge — use 50k rows to keep it fast on Codespaces
txn = pd.read_csv('../data/train_transaction.csv', nrows=50000)
idn = pd.read_csv('../data/train_identity.csv')
df = txn.merge(idn[['TransactionID','DeviceType','DeviceInfo']], on='TransactionID', how='left')
print(f'Rows: {len(df):,}  Fraud rate: {df.isFraud.mean():.2%}')

In [ ]:
# Build graph-ready columns
# source = card1 (proxy for cardholder), target = addr1 (merchant location)
df['source_node'] = 'card_' + df['card1'].astype(str)
df['target_node'] = 'addr_' + df['addr1'].fillna('unknown').astype(str)
df['timestamp']   = df['TransactionDT']

# Tabular features — curated small set
TAB_FEATURES = [
    'TransactionAmt',
    'C1','C2','C5','C6','C9','C11','C13','C14',
    'D1','D4','D10','D15',
    'V12','V13','V14','V17','V20','V23',
]
# Fill NaN
df[TAB_FEATURES] = df[TAB_FEATURES].fillna(0)
print('Graph nodes — source sample:', df['source_node'].value_counts().head())
print('Graph nodes — target sample:', df['target_node'].value_counts().head())

In [ ]:
# Temporal train/test split — critical: no future leakage
split_time = df['timestamp'].quantile(0.8)
train_df = df[df['timestamp'] <= split_time].copy()
test_df  = df[df['timestamp'] >  split_time].copy()

y_train = train_df['isFraud']
y_test  = test_df['isFraud']

print(f'Train: {len(train_df):,} rows, fraud rate: {y_train.mean():.2%}')
print(f'Test:  {len(test_df):,} rows,  fraud rate: {y_test.mean():.2%}')

In [ ]:
# Fit graphforest
model = DirectedGraphForestClassifier(
    source_col='source_node',
    target_col='target_node',
    timestamp_col='timestamp',
    amount_col='TransactionAmt',
    tabular_features=TAB_FEATURES,
    n_estimators=100,
)
model.fit(train_df, y_train)
print('Model fitted.')

In [ ]:
# Evaluate
proba = model.predict_proba(test_df)
fraud_proba = proba[:, list(model.classes_).index(1)] if 1 in model.classes_ else proba[:, 1]

auc  = roc_auc_score(y_test, fraud_proba)
ap   = average_precision_score(y_test, fraud_proba)
print(f'ROC-AUC:           {auc:.4f}')
print(f'Average Precision: {ap:.4f}')
print()
preds = model.predict(test_df)
print(classification_report(y_test, preds, target_names=['normal','fraud']))

In [ ]:
# Explain a fraud transaction
import json
fraud_idx = test_df[y_test == 1].index[0]
explainer = GraphForestExplainer(model)
explanation = explainer.explain(test_df, row_index=fraud_idx)
print(json.dumps(explanation, indent=2, default=str))

In [ ]:
# Graph RF feature importances
feat_imp = pd.Series(
    model._graph_rf.feature_importances_,
    index=model.graph_features
).sort_values(ascending=False)
print('Graph feature importances:')
print(feat_imp.round(4))